### 点云下载

In [ ]:
import open3d as o3d 
ds = o3d.data.PLYPointCloud()
print("Downloaded to:", ds.path)   # 输出实际文件路径

### 可视化

In [6]:

pcd = o3d.io.read_point_cloud("data/fragment.ply")
print(pcd)
print("Num points:", len(pcd.points))
print("Has colors:", pcd.has_colors())
print("Has normals:", pcd.has_normals())
o3d.visualization.draw_geometries([pcd])

PointCloud with 196133 points.
Num points: 196133
Has colors: True
Has normals: True
[Open3D WARNING] GLFW Error: Failed to detect any supported platform
[Open3D WARNING] GLFW initialized for headless rendering.
[Open3D WARNING] Failed to initialize GLEW.
[Open3D WARNING] [DrawGeometries] Failed creating OpenGL window.


### 转Numpy 方便降采样处理

In [ ]:
import numpy as np

points = np.asarray(pcd.points)   # (N,3)
colors = np.asarray(pcd.colors)   # (N,3) if exists
normals = np.asarray(pcd.normals) # (N,3) if exists
print(points.shape)
print(colors.shape)
print(normals.shape)



(196133, 3)
(196133, 3)
(196133, 3)


### Random Sample

In [ ]:
import numpy as np

def random_sample(points, mode=None,ratio=0.2, num_points=1, seed=None):
    """
    points: (N, D) 例如 D=3(xyz) 或 D=6(xyzrgb)
    ratio:  采样比例 (0,1]
    """
    N = points.shape[0]
    k = max(1, int(N * ratio)) if mode == "ratio" else max(1, int(num_points))
    rng = np.random.default_rng(seed)
    idx = rng.choice(N, size=k, replace=False)
    return points[idx], idx

sample_point,sample_point_idx=random_sample(points, mode="ratio", ratio=0.5)
pcd_down=o3d.geometry.PointCloud()
pcd_down.points=o3d.utility.Vector3dVector(sample_point)
pcd_down.colors=o3d.utility.Vector3dVector(colors[sample_point_idx])
o3d.visualization.draw_geometries([pcd_down])

### 体素降采样

In [ ]:
def uniform_sample(points, voxel_size=0.05, mode="centroid", seed=None):
    """
    Voxel (uniform) downsampling using hashing.

    points: (N, D)  D>=3, first 3 dims are xyz
    voxel_size: voxel edge length
    mode:
      - "first": keep the first point in each voxel
      - "random": keep a random point in each voxel
      - "centroid": keep centroid (mean xyz) per voxel (fast & stable)
    return:
      - sampled_points: (M, D) if mode in ["first","random"], (M,3) if centroid
      - indices: (M,) indices for first/random, None for centroid
    """
    pts = np.asarray(points)
    assert pts.ndim == 2 and pts.shape[1] >= 3

    xyz = pts[:, :3]
    # voxel coordinate (integer index per axis)
    voxel_idx = np.floor(xyz / voxel_size).astype(np.int64)

    # hash key for each voxel (tuple is simplest & safe)
    keys = [tuple(v) for v in voxel_idx]

    rng = np.random.default_rng(seed)

    if mode == "first":
        voxel_to_idx = {}
        for i, k in enumerate(keys):
            if k not in voxel_to_idx:
                voxel_to_idx[k] = i
        idx = np.fromiter(voxel_to_idx.values(), dtype=np.int64)
        return pts[idx], idx

    elif mode == "random":
        # collect indices per voxel
        voxel_to_list = {}
        for i, k in enumerate(keys):
            voxel_to_list.setdefault(k, []).append(i)
        chosen = []
        for k, lst in voxel_to_list.items():
            chosen.append(int(rng.choice(lst)))
        idx = np.array(chosen, dtype=np.int64)
        return pts[idx], idx

    elif mode == "centroid":
        # accumulate sum and count per voxel
        voxel_sum = {}
        voxel_cnt = {}
        for i, k in enumerate(keys):
            if k in voxel_sum:
                voxel_sum[k] += xyz[i]
                voxel_cnt[k] += 1
            else:
                voxel_sum[k] = xyz[i].copy()
                voxel_cnt[k] = 1
        centroids = np.stack([voxel_sum[k] / voxel_cnt[k] for k in voxel_sum.keys()], axis=0)
        return centroids, None

    else:
        raise ValueError("mode must be 'first', 'random', or 'centroid'")


### 最远距离降采样

In [ ]:
def farthest_point_sample(points, num_points, seed=None):
    N = points.shape[0]
    num_points = min(num_points, N)

    rng = np.random.default_rng(seed)
    first = int(rng.integers(0, N))

    centroids = np.zeros(num_points, dtype=np.int64)
    centroids[0] = first

    # dist[j] = 点 j 到“已选集合”的最小平方距离（初始化为无穷大）
    dist = np.full(N, np.inf)
    farthest = first

    for i in range(1, num_points):
        point = points[farthest]                       # 最新加入的点 (3,)
        d = np.sum((points - point) ** 2, axis=1)      # 所有点到它的平方距离 (N,)
        dist = np.minimum(dist, d)                     # 维护 min distance
        farthest = int(np.argmax(dist))                # 选 min-dist 最大的点
        centroids[i] = farthest

    return centroids


### 法线降采样

In [ ]:
def normal_space_sample(points, num_points=1024, seed=None, bins_theta=8, bins_phi=16):
    """
    Normal Space Sampling (NSS)

    points: (N,6) array -> [x,y,z,nx,ny,nz]
    num_points: number of samples
    bins_theta: bins for polar angle theta in [0, pi]
    bins_phi: bins for azimuth angle phi in [0, 2pi)
    return:
      - sampled_points: (K,6)
      - idx: (K,) indices in original points
    """
    assert points.ndim == 2 and points.shape[1] >= 6, \
        "points must be (N,6+) with normals in columns 3:6"

    rng = np.random.default_rng(seed)
    N = points.shape[0]
    K = min(num_points, N)

    normals = points[:, 3:6].astype(np.float64)
    # normalize normals (avoid zero division)
    norm = np.linalg.norm(normals, axis=1, keepdims=True)
    normals = normals / (norm + 1e-12)

    nx, ny, nz = normals[:, 0], normals[:, 1], normals[:, 2]

    # spherical angles
    # theta: [0, pi], phi: [0, 2pi)
    theta = np.arccos(np.clip(nz, -1.0, 1.0))
    phi = np.arctan2(ny, nx)
    phi = np.mod(phi, 2 * np.pi)

    # bin indices
    t_bin = np.floor(theta / np.pi * bins_theta).astype(np.int32)
    p_bin = np.floor(phi / (2 * np.pi) * bins_phi).astype(np.int32)
    t_bin = np.clip(t_bin, 0, bins_theta - 1)
    p_bin = np.clip(p_bin, 0, bins_phi - 1)

    bin_id = t_bin * bins_phi + p_bin
    num_bins = bins_theta * bins_phi

    # collect indices per bin
    bins = [[] for _ in range(num_bins)]
    for i in range(N):
        bins[bin_id[i]].append(i)

    # target per bin (roughly uniform in normal space)
    # First pass: allocate floor(K/num_bins) per non-empty bin
    base = K // num_bins
    selected = []

    # sample base points from each bin
    for b in range(num_bins):
        if len(bins[b]) == 0:
            continue
        m = min(base, len(bins[b]))
        if m > 0:
            chosen = rng.choice(bins[b], size=m, replace=False)
            selected.extend(chosen.tolist())

    # fill remaining by sampling from bins with leftover points
    remaining = K - len(selected)
    if remaining > 0:
        # build candidate pool of not-yet-selected indices, preferring bins with more leftovers
        selected_set = set(selected)
        leftovers = []
        for b in range(num_bins):
            if len(bins[b]) == 0:
                continue
            # add all remaining indices in this bin
            for idx in bins[b]:
                if idx not in selected_set:
                    leftovers.append(idx)

        if len(leftovers) > 0:
            take = min(remaining, len(leftovers))
            extra = rng.choice(np.array(leftovers), size=take, replace=False)
            selected.extend(extra.tolist())

    # if still short (rare), fallback random from whole set
    if len(selected) < K:
        missing = K - len(selected)
        pool = np.setdiff1d(np.arange(N), np.array(selected, dtype=np.int64), assume_unique=False)
        if len(pool) > 0:
            extra = rng.choice(pool, size=min(missing, len(pool)), replace=False)
            selected.extend(extra.tolist())

    idx = np.array(selected[:K], dtype=np.int64)
    return points[idx], idx
